# Hugging Face 모델 탐색 실습

이번 실습에서는 Hugging Face에서 제공하는 다양한 인공지능 모델을 직접 찾아보고, Colab에서 실행해보겠습니다.

오늘은 모델을 직접 학습시키는 것이 아니라, 이미 학습되어 공개된 모델을 불러와 사용하는 방법을 연습합니다.

이번 시간에 실습할 task는 다음 4가지입니다.

1. 감정분석
2. 번역
3. 한국어 요약
4. 문장 유사도 비교


#라이브러리 설치

Hugging Face 모델을 사용하기 위해 필요한 라이브러리를 먼저 설치하겠습니다.

아래 코드는 실습에 필요한 기본 라이브러리를 설치하는 코드입니다.
처음 한 번만 실행하면 됩니다.

transformers: Hugging Face 모델을 불러오고 실행하는 라이브러리입니다.
sentencepiece: 일부 번역 및 요약 모델에서 사용하는 토크나이저 라이브러리입니다.
sentence-transformers: 문장을 벡터로 바꾸고 유사도를 계산할 때 사용하는 라이브러리입니다.

In [ ]:
!pip install -q transformers sentencepiece sentence-transformers accelerate

In [ ]:
import transformers
import sentence_transformers

print("transformers:", transformers.__version__)
print("sentence-transformers:", sentence_transformers.__version__)

#Hugging Face 모델 검색 방법

Hugging Face에는 다양한 인공지능 모델이 공개되어 있습니다.

이번 실습에서는 원하는 task에 맞는 모델을 직접 검색해보고, Colab에서 실행해보겠습니다.

모델을 찾는 순서는 다음과 같습니다.

1. Hugging Face 사이트에 접속합니다.
2. 상단 메뉴에서 `Models`를 클릭합니다.
3. 왼쪽 메뉴에서 원하는 `Task`를 선택합니다.
4. 검색창에 원하는 키워드를 입력합니다.
5. 마음에 드는 모델 페이지에 들어갑니다.
6. 모델 이름을 복사합니다.
7. Colab 코드의 `model_name` 부분에 붙여넣습니다.

모델 이름은 보통 다음과 같은 형태입니다.

```text
사용자명/모델명
```

예를 들어 다음과 같은 이름을 사용할 수 있습니다.

```text
distilbert-base-uncased-finetuned-sst-2-english
eenzeenee/t5-base-korean-summarization
sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
```

처음에는 어떤 모델을 골라야 할지 헷갈릴 수 있습니다.
그럴 때는 다운로드 수가 많거나, 모델 카드에 사용 예시가 잘 적혀 있는 모델을 먼저 선택해보면 좋습니다.


#1. 감정분석
- sentiment-analysis
- text-classificatio

In [ ]:
#tabularisai/multilingual-sentiment-analysis
from transformers import pipeline

pipe = pipeline("text-classification", model="tabularisai/multilingual-sentiment-analysis")

In [ ]:
# 1. 분석하고 싶은 문장 입력
text = "이 영화 정말 재미있어요! 완전 추천합니다."

# 2. 감정 분석 실행
result = pipe(text)

# 3. 결과 출력
print(f"분석 결과: {result}")

#2. 번역
- translation

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("gogamza/kobart-summarization")
model = AutoModelForSeq2SeqLM.from_pretrained("gogamza/kobart-summarization")

In [ ]:
# 1. 번역하고 싶은 한국어 문장 입력
text = "오늘 날씨가 정말 좋네요. 같이 산책 가실래요?"

# 2. 모델이 요구하는 번역 양식(프롬프트)에 맞게 정렬
messages = [{"role": "user", "content": f"Translate the following Korean text into English:\n{text} <en>"}]


result = pipe_translate(messages, max_new_tokens=256, max_length=None, return_full_text=False)

# 4. 결과만 깔끔하게 출력
print(f"번역 결과: {result[0]['generated_text']}")


#3. 요약
- summarization
- text2text-generation

In [ ]:
import torch
from transformers import PreTrainedTokenizerFast
from transformers import BartForConditionalGeneration

# 1. 모델과 토크나이저 불러오기 (최초 1회만 실행)
tokenizer = PreTrainedTokenizerFast.from_pretrained('gogamza/kobart-summarization')
model = BartForConditionalGeneration.from_pretrained('gogamza/kobart-summarization')

In [ ]:
# 2. 요약할 인공지능 관련 텍스트 입력
text = "최근 인공지능 기술이 빠르게 발전하면서 일상생활의 다양한 분야에 AI가 도입되고 있습니다. 특히 자연어 처리 기술의 발달로 인해 기계 번역, 문서 요약, 챗봇 등의 서비스가 눈에 띄게 향상되었습니다. 전문가들은 향후 몇 년 안에 AI가 사람들의 업무 생산성을 크게 높여줄 것으로 전망하고 있으며, 관련 산업의 규모도 폭발적으로 성장할 것으로 기대하고 있습니다."

# 3. 시작(BOS)과 끝(EOS) 토큰을 명확히 넣어주어 환각 방지
raw_input_ids = tokenizer.encode(text)
input_ids = [tokenizer.bos_token_id] + raw_input_ids + [tokenizer.eos_token_id]

# 4. 요약문 생성
summary_ids = model.generate(torch.tensor([input_ids]), max_length=64, num_beams=4, early_stopping=True)

# 5. 결과를 디코딩하고 거슬리는 밑줄(_) 기호를 깔끔하게 세탁
raw_summary = tokenizer.decode(summary_ids.squeeze().tolist(), skip_special_tokens=True)
clean_summary = raw_summary.replace(" ", " ").replace("__", " ").replace("_", " ").strip()
final_summary = " ".join(clean_summary.split())

print(f"원문 글자 수: {len(text)}자")
print(f"요약 결과: {final_summary}")

#4. 문장 유사도 비교
- sentence-similarity
- sentence-transformers 라이브러리 사용

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/static-similarity-mrl-multilingual-v1")

sentences = [
    "A man is jumping unto his filthy bed.",
    "A man is ouside near the beach.",
    "The bed is dirty.",
    "The man is on the moon."
]
embeddings = model.encode(sentences)

similarities = model.similarity(embeddings, embeddings)
print(similarities.shape)

In [ ]:

print("\n[문장 유사도 분석 결과]")

# 첫 번째 문장("남자가 더러운 침대로 뛰어든다")을 기준으로 설정합니다.
base_idx = 0
print(f"👉 기준 문장: '{sentences[base_idx]}'\n")

# 나머지 3개의 문장들과 유사도를 비교합니다.
for i in range(1, len(sentences)):
    # 데이터에서 숫자만 빼내어 소수점 4자리까지만 깔끔하게 자릅니다.
    score = similarities[base_idx][i].item()

    print(f"비교: '{sentences[i]}'")
    print(f"점수: {score:.4f} (1.0에 가까울수록 의미가 비슷함)\n")